# MOD-03 · NB-07 — Confounding, Effect Modification & Multivariable Adjustment
### Health Analytics with Python · Module 03: Statistical Inference
---
**Learning objectives**
- Distinguish confounding from effect modification (interaction) conceptually and empirically
- Use DAGs to identify adjustment sets
- Test for effect modification with interaction terms
- Apply stratification and regression adjustment to control confounding
- Compute E-values as a sensitivity analysis for unmeasured confounding

**Estimated time:** 2 hours | **Level:** Advanced | **Libraries:** `pandas`, `numpy`, `statsmodels`, `matplotlib`


## 1. Setup & Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')
import os; os.makedirs('/tmp/mod03',exist_ok=True)

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

np.random.seed(42); N=3000
def logistic(x): return 1/(1+np.exp(-x))

# ── Age is a confounder: older → more diabetes AND more readmission ────────────
age = np.random.normal(62,15,N).clip(18,95).astype(int)
age_std = (age - 62) / 15

diabetes = np.random.binomial(1, logistic(-1.0 + 0.05*age_std), N)
sex      = np.random.choice(['M','F'],N,p=[0.48,0.52])
payer    = np.random.choice(['Medicare','Medicaid','Private'],N,p=[0.42,0.22,0.36])
los_days = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
hf       = np.random.binomial(1,logistic(-1.8+0.6*diabetes+0.04*age_std),N)

# Readmission: diabetes → readmission but age is a confounder
readmit_logit = (-2.8 + 0.5*diabetes + 0.03*age_std + 0.6*hf
                 + 0.2*(payer=='Medicaid').astype(int)
                 + 0.15*(los_days>7).astype(int)
                 + np.random.normal(0,0.2,N))
readmitted = np.random.binomial(1, logistic(readmit_logit), N)

# Effect modifier: diabetes effect on readmission is STRONGER in patients with HF
readmit_em_logit = (-2.8 + 0.3*diabetes + 0.03*age_std + 0.6*hf
                    + 0.9*diabetes*hf              # interaction term
                    + 0.2*(payer=='Medicaid').astype(int)
                    + np.random.normal(0,0.2,N))
readmitted_em = np.random.binomial(1, logistic(readmit_em_logit), N)

df = pd.DataFrame({
    'patient_id': [f'PT-{i:05d}' for i in range(1,N+1)],
    'age':age,'age_std':age_std,'sex':sex,'payer':payer,
    'los_days':los_days,'diabetes':diabetes,'hf':hf,
    'readmitted':readmitted,'readmitted_em':readmitted_em,
})
df['age_group'] = pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])

print(f"Dataset: {df.shape}")
print(f"Diabetes prevalence   : {diabetes.mean()*100:.1f}%")
print(f"Readmission prevalence: {readmitted.mean()*100:.1f}%")


## 2. What is Confounding?

A **confounder** is a variable that is associated with both the exposure and the outcome, but is **not** on the causal pathway between them.

```
        Age
       ↙    ↘
Diabetes → Readmission
```

Age → more diabetes (older patients have more diabetes)  
Age → more readmission (older patients readmit more)  
Age is NOT caused by diabetes  

**Effect**: crude OR for diabetes → readmission is inflated because age is "lurking" in the background.


In [ ]:
# ── Demonstrate confounding: crude vs adjusted estimate ──────────────────────
from scipy.stats import chi2_contingency

def crude_or(exposed, outcome):
    a = ((exposed==1)&(outcome==1)).sum()
    b = ((exposed==1)&(outcome==0)).sum()
    c = ((exposed==0)&(outcome==1)).sum()
    d = ((exposed==0)&(outcome==0)).sum()
    OR = (a*d)/(b*c) if b*c>0 else np.nan
    se = np.sqrt(1/a+1/b+1/c+1/d)
    return OR, np.exp(np.log(OR)-1.96*se), np.exp(np.log(OR)+1.96*se)

# Crude
cr_or, cr_lo, cr_hi = crude_or(df.diabetes, df.readmitted)

# Adjusted (logistic regression)
model_adj = smf.logit(
    'readmitted ~ diabetes + age_std + C(sex) + C(payer,"Private") + hf + los_days',
    data=df
).fit(disp=0)
adj_or_val = np.exp(model_adj.params['diabetes'])
adj_ci = np.exp(model_adj.conf_int().loc['diabetes'])

print("Diabetes → 30-day readmission")
print(f"  Crude OR (unadjusted): {cr_or:.3f}  (95% CI: {cr_lo:.3f}–{cr_hi:.3f})")
print(f"  Adjusted OR           : {adj_or_val:.3f}  (95% CI: {adj_ci[0]:.3f}–{adj_ci[1]:.3f})")
print()
pct_change = (adj_or_val - cr_or)/cr_or * 100
print(f"  % change after adjustment: {pct_change:+.1f}%")
print()
if abs(pct_change) > 10:
    print("  ✗ Meaningful confounding detected (>10% change rule)")
else:
    print("  ✓ Little confounding (< 10% change)")


## 3. Directed Acyclic Graph (DAG)

In [ ]:
# Draw a simple DAG illustrating confounding and adjustment
fig, axes = plt.subplots(1,2,figsize=(14,5))

def draw_dag(ax, title, nodes, edges, highlight_edges=None):
    ax.set_xlim(0,10); ax.set_ylim(0,6); ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')
    node_pos = nodes
    for name,(x,y) in node_pos.items():
        color = '#FFD580' if name=='Age' else ('#D65F5F' if name in ['Diabetes','Readmission'] else '#AEC6CF')
        circle = plt.Circle((x,y),0.7,color=color,zorder=3,linewidth=1.5,edgecolor='gray')
        ax.add_patch(circle)
        ax.text(x,y,name,ha='center',va='center',fontsize=9,fontweight='bold',zorder=4)
    for (src,dst,style) in edges:
        x1,y1 = node_pos[src]; x2,y2 = node_pos[dst]
        dx,dy = x2-x1,y2-y1
        length = np.sqrt(dx**2+dy**2)
        x1n,y1n = x1+0.7*dx/length,y1+0.7*dy/length
        x2n,y2n = x2-0.8*dx/length,y2-0.8*dy/length
        color = 'red' if highlight_edges and (src,dst) in highlight_edges else '#333'
        ls = '--' if style=='dashed' else '-'
        ax.annotate('',xy=(x2n,y2n),xytext=(x1n,y1n),
                    arrowprops=dict(arrowstyle='->',color=color,lw=1.8,linestyle=ls))

nodes = {'Diabetes':(2,3),'Readmission':(8,3),'Age':(5,5.5),'Heart Failure':(5,1)}
edges = [('Diabetes','Readmission','solid'),('Age','Diabetes','solid'),
         ('Age','Readmission','solid'),('Diabetes','Heart Failure','solid'),
         ('Heart Failure','Readmission','solid')]
draw_dag(axes[0],'DAG — Confounding by Age',nodes,edges,highlight_edges=[('Age','Diabetes'),('Age','Readmission')])
axes[0].text(2,0.3,'Yellow=confounder  |  Red arrows=confounding paths',fontsize=8,color='gray')

# DAG with effect modification
nodes2 = {'Diabetes':(2,3),'Readmission':(8,3),'Heart Failure':(5,1),'Age':(5,5.5)}
draw_dag(axes[1],'DAG — Effect Modification (HF modifies Diabetes effect)',nodes2,edges,
         highlight_edges=[('Heart Failure','Readmission')])
axes[1].annotate('',xy=(7.3,2.5),xytext=(5.7,1.5),
                arrowprops=dict(arrowstyle='->',color='purple',lw=2,
                                connectionstyle='arc3,rad=0.3'))
axes[1].text(6.2,1.9,'HF modifies
DM effect',ha='center',fontsize=8,color='purple')

plt.tight_layout()
plt.savefig('/tmp/mod03/dag.png',bbox_inches='tight'); plt.show()


## 4. Effect Modification (Interaction)

In [ ]:
# Test whether HF MODIFIES the effect of diabetes on readmission
# (as opposed to HF merely being a confounder)

# Model without interaction
model_no_int = smf.logit(
    'readmitted_em ~ diabetes + hf + age_std + C(payer,"Private")',
    data=df
).fit(disp=0)

# Model with interaction
model_int = smf.logit(
    'readmitted_em ~ diabetes * hf + age_std + C(payer,"Private")',
    data=df
).fit(disp=0)

# Likelihood ratio test for interaction term
lr_stat = -2*(model_no_int.llf - model_int.llf)
from scipy.stats import chi2 as chi2_dist
lr_p    = 1 - chi2_dist.cdf(lr_stat, df=1)

print("Likelihood ratio test for diabetes × HF interaction:")
print(f"  LR statistic = {lr_stat:.3f}, df=1, p = {lr_p:.4f}")
print()
if lr_p < 0.05:
    print("  ✓ Significant interaction — effect modification is present")
    print("  → Report STRATUM-SPECIFIC effects, not a single pooled estimate")
else:
    print("  ✗ No significant interaction — no effect modification")
    print("  → A single adjusted estimate is appropriate")

# Stratum-specific ORs
print("\nStratum-specific OR for diabetes → readmission:")
for hf_val,label in [(0,'No HF'),(1,'HF present')]:
    sub = df[df['hf']==hf_val]
    a = ((sub.diabetes==1)&(sub.readmitted_em==1)).sum()
    b = ((sub.diabetes==1)&(sub.readmitted_em==0)).sum()
    c = ((sub.diabetes==0)&(sub.readmitted_em==1)).sum()
    d = ((sub.diabetes==0)&(sub.readmitted_em==0)).sum()
    or_s = (a*d)/(b*c) if b*c>0 else np.nan
    se   = np.sqrt(1/a+1/b+1/c+1/d)
    lo,hi = np.exp(np.log(or_s)-1.96*se),np.exp(np.log(or_s)+1.96*se)
    print(f"  {label:15s}: OR = {or_s:.2f}  (95% CI: {lo:.2f}–{hi:.2f})")


In [ ]:
# Visualise effect modification
fig, axes = plt.subplots(1,2,figsize=(13,5))

# Panel A — bar chart of stratum-specific rates
strat_rates = df.groupby(['hf','diabetes'])['readmitted_em'].mean().unstack()*100
strat_rates.index = ['No HF','HF']
strat_rates.columns = ['No diabetes','Diabetes']
strat_rates.plot(kind='bar',ax=axes[0],color=['#4878CF','#D65F5F'],
                 edgecolor='white',rot=0)
axes[0].set_ylabel('Readmission rate (%)'); axes[0].set_xlabel('')
axes[0].set_title('A  Effect modification: DM × HF
(parallel bars = no interaction; non-parallel = interaction)')
axes[0].legend(title='Diabetes status',fontsize=9)

# Panel B — interaction plot
for hf_val,color,label in [(0,'#4878CF','No HF'),(1,'#D65F5F','HF present')]:
    sub = df[df['hf']==hf_val]
    rates = sub.groupby('diabetes')['readmitted_em'].mean()*100
    axes[1].plot([0,1],rates.values,marker='o',ms=10,lw=2.5,color=color,label=label)

axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['No diabetes','Diabetes'])
axes[1].set_ylabel('Readmission rate (%)'); axes[1].set_xlabel('')
axes[1].set_title('B  Interaction plot
(crossing/diverging lines = effect modification)')
axes[1].legend(title='Heart failure',fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/mod03/effect_modification.png',bbox_inches='tight'); plt.show()


## 5. The 10% Change Rule for Confounding

In [ ]:
print("Systematic 10% change rule across all predictors\n")
print(f"{'Predictor':20s} {'Crude OR':>10s} {'Adj OR':>10s} {'% Change':>10s} {'Confounder?':>12s}")
print("-"*65)

outcome_col = 'readmitted'
for exposure, label in [
    ('diabetes','Diabetes'),('hf','Heart failure'),
    ('age_std','Age (std)'),
]:
    # Crude
    if exposure == 'age_std':
        m_crude = smf.logit(f'{outcome_col} ~ {exposure}', data=df).fit(disp=0)
        cr = np.exp(m_crude.params[exposure])
    else:
        a = ((df[exposure]==1)&(df[outcome_col]==1)).sum()
        b = ((df[exposure]==1)&(df[outcome_col]==0)).sum()
        c = ((df[exposure]==0)&(df[outcome_col]==1)).sum()
        d = ((df[exposure]==0)&(df[outcome_col]==0)).sum()
        cr = (a*d)/(b*c) if b*c>0 else np.nan

    # Adjusted (controlling for the other two)
    covariates = [v for v in ['diabetes','hf','age_std'] if v != exposure]
    formula_a  = f'{outcome_col} ~ {exposure} + {" + ".join(covariates)}'
    m_adj = smf.logit(formula_a, data=df).fit(disp=0)
    adj   = np.exp(m_adj.params[exposure])

    pct_change = (adj - cr)/cr * 100
    is_conf    = '✗ Yes (>10%)' if abs(pct_change) > 10 else '✓ No (<10%)'
    print(f"{label:20s} {cr:>10.3f} {adj:>10.3f} {pct_change:>+9.1f}% {is_conf:>12s}")


## 6. E-Value: Sensitivity to Unmeasured Confounding

In [ ]:
def e_value(RR):
    """
    E-value: minimum strength of association an unmeasured confounder would need
    to fully explain an observed RR, on both the exposure-confounder and
    confounder-outcome pathways.
    VanderWeele & Ding (2017) formula.
    """
    if RR < 1:
        RR = 1/RR   # flip if protective
    return RR + np.sqrt(RR * (RR - 1))

# Convert OR to approximate RR using prevalence correction
outcome_prev = df['readmitted'].mean()
def or_to_rr(OR, prev):
    """Zhang & Yu (1998) approximation."""
    return OR / ((1 - prev) + prev * OR)

print("E-value analysis — sensitivity to unmeasured confounding")
print("="*65)
for label, OR_val in [
    ('Diabetes → readmission (adj OR)', adj_or_val),
    ('HF → readmission', np.exp(model_adj.params['hf'])),
]:
    RR_approx = or_to_rr(OR_val, outcome_prev)
    ev = e_value(RR_approx)
    ev_ci = e_value(max(1.0, RR_approx / np.exp(1.96*0.15)))  # rough CI lower
    print(f"\n{label}")
    print(f"  Adjusted OR ≈ RR  : {RR_approx:.3f}")
    print(f"  E-value (point)   : {ev:.2f}")
    print(f"  E-value (CI lower): {ev_ci:.2f}")
    print(f"  → An unmeasured confounder would need to be associated with BOTH")
    print(f"    exposure AND outcome by a factor of at least {ev:.1f}× to explain away")
    print(f"    the observed association entirely.")
    if ev > 2.0:
        print(f"  → E-value > 2 suggests robustness to weak confounding.")
    else:
        print(f"  → E-value ≤ 2 — association could be explained by a moderate confounder.")


## 7. Knowledge Check
1. In the DAG, age is a confounder. List two things you must verify for it to be a valid confounder (not a collider or mediator).
2. The diabetes OR changes from 1.82 (crude) to 1.31 (adjusted) — a 28% reduction. What does this mean?
3. The LR test for diabetes × HF interaction gives p = 0.002. What are the reporting implications?
4. The E-value for the adjusted OR is 2.4. What would this mean for a clinical audience?
5. Add sex as a potential effect modifier for the diabetes → readmission association. Is there evidence of differential effects by sex?


In [ ]:
# Exercise 5 — sex as effect modifier
model_sex_int = smf.logit(
    'readmitted ~ diabetes * C(sex) + hf + age_std + C(payer,"Private")',
    data=df
).fit(disp=0)
lr_sex = -2*(model_adj.llf - model_sex_int.llf)
p_sex  = 1 - chi2_dist.cdf(lr_sex, df=1)
print(f"LR test for diabetes × sex interaction: p = {p_sex:.4f}")
print()
print("Stratum-specific OR by sex:")
for sex_val in ['M','F']:
    sub = df[df['sex']==sex_val]
    a = ((sub.diabetes==1)&(sub.readmitted==1)).sum()
    b = ((sub.diabetes==1)&(sub.readmitted==0)).sum()
    c = ((sub.diabetes==0)&(sub.readmitted==1)).sum()
    d = ((sub.diabetes==0)&(sub.readmitted==0)).sum()
    or_s = (a*d)/(b*c)
    print(f"  {sex_val}: OR = {or_s:.3f}")


---
**Next → NB-08: Capstone — Multivariable Inference Pipeline**